# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,MSFT,16,6,0.102400,2026-07-14 16:29:28+00:00
1,META,9,5,0.130680,2026-07-11 02:39:00+00:00
2,IBM,9,5,0.262940,2026-07-14 22:54:47+00:00
3,NFLX,9,5,0.378700,2026-07-14 18:42:35+00:00
4,SPY,7,5,0.079380,2026-07-13 22:12:26+00:00
5,SPCX,4,3,0.008367,2026-07-12 16:09:56+00:00
6,AFL,11,1,0.995300,2026-07-13 15:32:51+00:00
7,BE,7,2,0.932750,2026-07-14 13:50:15+00:00
8,HBM,3,3,0.579733,2026-07-09 23:10:14+00:00
9,RAM,5,2,0.474950,2026-07-13 18:12:52+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['MSFT', 'META', 'IBM', 'NFLX']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    col_data = in_df[column_name].dropna()
    if col_data.empty:
        return None
    return caster(col_data.iloc[-1])


price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": safe_latest_value(features_df, "Close", float),
        "latest_rsi": safe_latest_value(features_df, "RSI", float),
        "latest_macd": safe_latest_value(features_df, "MACD", float),
        "return_21d": safe_latest_value(features_df, "21D Return", float),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,NFLX,251,73.529999,38.887196,-2.179363,-0.095238,1.590013,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== MSFT latest metrics =====
                 Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                     
2026-07-08  383.339996  25908300           26.568560          397.770996           405.728361  46.525637 -6.815175   0.027666   -0.079991
2026-07-09  384.359985  31083200           26.446950          396.715329           405.238717  47.090442 -6.178637   0.000208   -0.066498
2026-07-10  385.100006  24644600           26.352270          395.796329           404.777284  47.523505 -5.550480  -0.013803   -0.045388
2026-07-13  390.989990  28914900           25.694275          394.596329           404.461403  50.963745 -4.525225   0.010989   -0.016031
2026-07-14  384.929993  27741653           23.489336          392.419329           404.013949  47.512507 -4.153812  -0.010056   -0.013860



## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== MSFT regression results =====
                 Close  Predicted Close
Date                                   
2026-06-30  373.019989       363.802922
2026-07-01  384.279999       373.110029
2026-07-02  390.489990       384.540331
2026-07-06  386.739990       379.900033
2026-07-07  388.839996       383.903424
2026-07-08  383.339996       380.679037
2026-07-09  384.359985       378.550993
2026-07-10  385.100006       379.369870
2026-07-13  390.989990       385.093043
2026-07-14  384.929993       380.624637

===== META regression results =====
                 Close  Predicted Close
Date                                   
2026-06-30  563.289978       558.554063
2026-07-01  612.909973       603.825464
2026-07-02  582.900024       586.986806
2026-07-06  600.289978       598.610753
2026-07-07  615.580017       609.727428
2026-07-08  603.119995       605.470838
2026-07-09  631.479980       619.333553
2026-07-10  669.210022       649.965215
2026-07-13  656.729980       646.312570
2026-0

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "MSFT",
    "META",
    "IBM",
    "NFLX"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
